In [ ]:
from langchain_openai.chat_models.base import ChatOpenAI
from app.configs.settings import settings

llm = ChatOpenAI(model="gpt-4o-mini", api_key=settings.OPENAI_API_KEY)

print("LLM configured", llm)

LLM configured metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18', 'langchain-openai': '1.6.0'}} profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True} client=<openai.resources.chat.completions.completions.Completions object at 0x000002BD1E76A510> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002BD1E76AF90> root_client=<openai.OpenAI object at 0x000002BD1DED86E0

In [4]:
result = llm.invoke("Hello")

print(result.content)
print(type(result))

Hello! How can I assist you today?
<class 'langchain_core.messages.ai.AIMessage'>


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage(content="You are a helpful ai assistant"),
    HumanMessage(content="What is 5 + 7?")
]

result = llm.invoke(messages)
print(result.content)

messages.append(result)
messages.append(HumanMessage(content="Now multiply that result by 2."))

response = llm.invoke(messages)
print(response.content)


5 + 7 equals 12.
12 multiplied by 2 equals 24.


In [7]:
for chunk in llm.stream("what is python?"):
    print(chunk.content, end="", flush=True)

Python is a high-level, interpreted programming language known for its simplicity and readability. It was created by Guido van Rossum and first released in 1991. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming.

Key features of Python include:

1. **Easy to Read and Write**: Python's syntax is designed to be clear and straightforward, making it accessible for beginners while allowing experienced developers to write code efficiently.

2. **Rich Standard Library**: Python comes with a large standard library that provides modules and tools for handling various tasks, such as file I/O, system calls, and data manipulation.

3. **Cross-Platform**: Python runs on various platforms, including Windows, macOS, and Linux, allowing developers to write code that can be executed anywhere.

4. **Extensive Community and Ecosystem**: Python has a large and active community that contributes to a vast collection of third-party libraries an

In [8]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful translator. Translate {input_language} to {output_language}."),
    ("human", "{text}")
])

formatted_messages = prompt.invoke({
    "input_language": "English",
    "output_language": "French",
    "text": "Hello, How are you?"
})

print(formatted_messages.to_messages())

[SystemMessage(content='You are a helpful translator. Translate English to French.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, How are you?', additional_kwargs={}, response_metadata={})]


In [10]:
# Pipe Operator (LCEL)

chain = prompt | llm

result = chain.invoke({
    "input_language": "English",
    "output_language": "French",
    "text": "Hello, How are you?"
})

print(result.content)
print(type(result))

Bonjour, comment ça va ?
<class 'langchain_core.messages.ai.AIMessage'>


In [21]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

chain = prompt | llm | StrOutputParser()

result = chain.invoke({
    "input_language": "English",
    "output_language": "French",
    "text": "Hello, How are you?"
})

print(result)
print(type(result))

I'm just a computer program, but I'm here and ready to help! How about a joke to lighten the mood? 

Why did the scarecrow win an award? 

Because he was outstanding in his field!
<class 'langchain_core.messages.base.TextAccessor'>


In [16]:
chain = prompt | llm | StrOutputParser()

for chunk in chain.stream({
    "input_language": "English",
    "output_language": "French",
    "text": "Hello, How are you?"
}):
    print(chunk, end="", flush=True)

Bonjour, comment ça va ?

In [ ]:
from pydantic import BaseModel

class Joke(BaseModel):
    joke: str
    punchline: str

structured_llm = llm.with_structured_output(Joke)

result: Joke = structured_llm.invoke("Tell any joke?")

print(result)
print(type(result))
print("Joke Baseline", result.joke)
print("Punchline", result.punchline)

joke="Why don't scientists trust atoms?" punchline='Because they make up everything!'
<class '__main__.Joke'>
Joke Baseline Why don't scientists trust atoms?
Punchline Because they make up everything!


In [27]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Tell me a joke and its punchline."),
    ("human", "{text}")
])

chain = prompt | structured_llm 

result: Joke = chain.invoke({"text": "Tell me a joke"})
print(result)

joke="Why don't scientists trust atoms?" punchline='Because they make up everything!'


In [31]:
for chunk in structured_llm.stream("tell me a joke"):
    print(chunk)

joke='Why did the scarecrow win an award?' punchline='Because he was outstanding in his field!'


e:\class\ai-class\backend\venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Joke(joke='Why did the sc...standing in his field!'), input_type=Joke])
  return self.__pydantic_serializer__.to_python(


In [ ]:
from langchain_core.tools import tool

@tool(description="Get the current weather for a given city.")
def get_weather(city: str) -> str:
    return f"The weather in {city} is sunny, 24°C degrees and clear"

llm_with_tools = llm.bind_tools([get_weather])

response = llm_with_tools.invoke("What's the weather in Tokyo?")
print(response.tool_calls)

[{'name': 'get_weather', 'args': {'city': 'Tokyo'}, 'id': 'call_7Kbrbdv1PrRLWUPoEDTzIC4k', 'type': 'tool_call'}]


In [35]:
from langchain_core.messages import ToolMessage, HumanMessage

messages = [HumanMessage(content="What's the weather in Tokyo?")]

result = llm_with_tools.invoke(messages)
messages.append(result)

for tool_call in result.tool_calls:
    tool_output = get_weather.invoke(tool_call["args"])
    messages.append(ToolMessage(content=tool_output, tool_call_id=tool_call["id"]))

final_response = llm_with_tools.invoke(messages)
print(final_response.content)

The weather in Tokyo is sunny, with a temperature of 24°C and clear skies.


In [39]:
from langchain_classic.agents.agent import AgentExecutor
from langchain_classic.agents.tool_calling_agent.base import create_tool_calling_agent
from langchain_core.prompts import MessagesPlaceholder

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, [get_weather], agent_prompt)

agent_executor = AgentExecutor(agent=agent, tools=[get_weather], verbose=True)

result = agent_executor.invoke({"input": "What's the weather in Tokyo?"})
print(result)




> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'city': 'Tokyo'}`


The weather in Tokyo is sunny, 24°C degrees and clearThe weather in Tokyo is sunny with a temperature of 24°C and clear skies.

> Finished chain.
{'input': "What's the weather in Tokyo?", 'output': 'The weather in Tokyo is sunny with a temperature of 24°C and clear skies.'}
